In [6]:
import numpy as np
from scipy.io import loadmat
import pandas as pd
import os

In [7]:
exclude_area = pd.read_csv("/media/ubuntu/sda/TrippleN/exclude_area.csv")
GoodUnit_session_id = sorted(os.listdir("/media/ubuntu/sda/TrippleN/GoodUnit"))
Processed_session_id = sorted(os.listdir("/media/ubuntu/sda/TrippleN/Processed"))
neuron_psth_session_id = sorted(os.listdir("/media/ubuntu/sda/TrippleN/psth"))
def extract_info(filename):
    parts = filename.replace('.mat', '').split('_')
    date = parts[1]
    subject = parts[2]
    return date, subject

session_info = {}
for session_id in GoodUnit_session_id:
    date, subject = extract_info(session_id)
    session_info[session_id] = {'date': date, 'subject': subject}

dates = []
subjects = []
goodunit_ids = []
processed_ids = []
neuron_psth_ids = []

for idx, row in exclude_area.iterrows():
    ses_idx = row['SesIdx']
    if ses_idx - 1 < len(GoodUnit_session_id):
        session_id = GoodUnit_session_id[ses_idx - 1]
        goodunit_ids.append(session_id)
        processed_ids.append(Processed_session_id[ses_idx - 1] if ses_idx - 1 < len(Processed_session_id) else None)
        neuron_psth_ids.append(neuron_psth_session_id[ses_idx - 1] if ses_idx - 1 < len(neuron_psth_session_id) else None)
        info = session_info[session_id]
        dates.append(info['date'])
        subjects.append(info['subject'])
    else:
        goodunit_ids.append(None)
        processed_ids.append(None)
        neuron_psth_ids.append(None)
        dates.append(None)
        subjects.append(None)

exclude_area['date'] = dates
exclude_area['subject'] = subjects
exclude_area['GoodUnit_session_id'] = goodunit_ids
exclude_area['Processed_session_id'] = processed_ids
exclude_area['neuron_psth_session_id'] = neuron_psth_ids


In [8]:
import pickle
import warnings
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

INTERESTED_AREA = ['MB', 'AB', 'MF', 'AF', 'MO', 'AO', 'LPP', 'PITP', 'CLC', 'AMC']
RELIABILITY_THRES = 0.4

def arealabel_to_coarse(arealabel):
    if pd.isna(arealabel) or arealabel == 'Unknown':
        return 'Unknown'
    for coarse in INTERESTED_AREA:
        if str(arealabel).startswith(coarse):
            return coarse
    return 'Unknown'

def assign_units_to_areas(pos, ses_idx, exclude_df):
    n_units = len(pos)
    arealabels = ['Unknown'] * n_units
    area_rows = exclude_df[exclude_df['SesIdx'] == ses_idx]
    for _, ar in area_rows.iterrows():
        if str(ar['AREALABEL']).lower() == 'unknown':
            continue
        y1, y2 = float(ar['y1']), float(ar['y2'])
        mask = (pos > y1) & (pos < y2)
        for i in np.where(mask)[0]:
            arealabels[i] = ar['AREALABEL']
    return arealabels

base_path = "/media/ubuntu/sda/TrippleN"
output_unit_path = os.path.join(base_path, "customize/unit_info")
output_psth_path = os.path.join(base_path, "customize/psth")
os.makedirs(output_unit_path, exist_ok=True)
os.makedirs(output_psth_path, exist_ok=True)

n_sessions = min(len(GoodUnit_session_id), len(Processed_session_id), len(neuron_psth_session_id))
session_list = []
for i in range(n_sessions):
    ses_idx = i + 1
    session_id = GoodUnit_session_id[i]
    info = session_info[session_id]
    session_list.append({
        'SesIdx': ses_idx,
        'GoodUnit_session_id': session_id,
        'Processed_session_id': Processed_session_id[i],
        'neuron_psth_session_id': neuron_psth_session_id[i],
        'date': info['date'],
        'subject': info['subject']
    })
print(f"Sessions to process: {len(session_list)}")

GoodUnit_folder = os.path.join(base_path, "GoodUnit")
Processed_folder = os.path.join(base_path, "Processed")
psth_folder = os.path.join(base_path, "psth")
extracted_folder = os.path.join(base_path, "customize/GoodUnitStr")

processed_columns = ['B_SI', 'F_SI', 'O_SI', 'UnitType', 'best_r_time1', 'best_r_time2',
                     'pos', 'reliability_basic', 'reliability_best', 'reliability_find_testset',
                     'snr', 'snrmax']

session_count = 0

for row in tqdm(session_list, desc='Sessions'):
    goodunit_file = row['GoodUnit_session_id']
    processed_file = row['Processed_session_id']
    psth_file = row['neuron_psth_session_id']
    ses_idx = row['SesIdx']

    if pd.isna(goodunit_file) or pd.isna(processed_file) or pd.isna(psth_file):
        continue

    processed_path = os.path.join(Processed_folder, processed_file)
    psth_path = os.path.join(psth_folder, psth_file)

    extracted_name = goodunit_file.replace('.mat', '')
    waveform_path = os.path.join(extracted_folder, f'{extracted_name}_waveform.npy')
    spikepos_path = os.path.join(extracted_folder, f'{extracted_name}_spikepos.npy')

    if not os.path.exists(processed_path) or not os.path.exists(psth_path):
        continue
    if not os.path.exists(waveform_path) or not os.path.exists(spikepos_path):
        continue

    try:
        waveform_data = np.load(waveform_path)
        spikepos_data = np.load(spikepos_path)
        psth_data = np.load(psth_path)

        n_neurons = waveform_data.shape[0]
        processed_mat = loadmat(processed_path)

        neuron_df = pd.DataFrame()
        for col in processed_columns:
            if col in processed_mat:
                data = processed_mat[col]
                if data.ndim == 2 and data.shape[0] == 1:
                    neuron_df[col] = data.flatten()
                else:
                    neuron_df[col] = data.flatten() if data.ndim > 1 else np.array([data]).flatten()
            else:
                neuron_df[col] = None

        pos = neuron_df['pos'].values
        arealabels = assign_units_to_areas(pos, ses_idx, exclude_area)
        areas = [arealabel_to_coarse(a) for a in arealabels]

        neuron_df['session_id'] = goodunit_file
        neuron_df['date'] = row['date']
        neuron_df['subject'] = row['subject']
        neuron_df['SesIdx'] = ses_idx
        neuron_df['AREALABEL'] = arealabels
        neuron_df['Area'] = areas

        neuron_df['waveform'] = [waveform_data[i] for i in range(n_neurons)]
        neuron_df['spikepos_1'] = spikepos_data[:, 0]
        neuron_df['spikepos_2'] = spikepos_data[:, 1]

        if len(neuron_df) == psth_data.shape[0]:
            reliability_best = neuron_df['reliability_best'].values
            non_unknown = (neuron_df['Area'].astype(str) != 'Unknown')
            mask = (reliability_best > RELIABILITY_THRES) & non_unknown
            filtered_neuron_df = neuron_df[mask].reset_index(drop=True)
            filtered_psth_data = psth_data[mask]

            session_count += 1
            session_name = extracted_name

            unit_info_file = os.path.join(output_unit_path, f'{session_name}_unit_info.pkl')
            with open(unit_info_file, 'wb') as f:
                pickle.dump(filtered_neuron_df, f)

            psth_file_out = os.path.join(output_psth_path, f'{session_name}_psth.npy')
            np.save(psth_file_out, filtered_psth_data)

    except Exception as e:
        continue

print(f"Processed {session_count} sessions, reliability_basic > {RELIABILITY_THRES}")


Sessions to process: 90


Sessions: 100%|██████████| 90/90 [14:12<00:00,  9.47s/it]

Processed 90 sessions, reliability_basic > 0.4


In [9]:
import pickle
import warnings
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

base_path = "/media/ubuntu/sda/TrippleN"
unit_info_folder = os.path.join(base_path, "customize/unit_info")
psth_folder = os.path.join(base_path, "customize/psth")
output_folder = os.path.join(base_path, "customize")

unit_info_files = [f for f in os.listdir(unit_info_folder) if f.endswith('_unit_info.pkl')]
psth_files = [f for f in os.listdir(psth_folder) if f.endswith('_psth.npy')]

subjects = set()
for f in unit_info_files:
    for subj in ['JianJian', 'FaCai', 'TuTu', 'ZhuangZhuang', 'MaoDan']:
        if subj in f:
            subjects.add(subj)
            break

print(f"Found {len(subjects)} subjects: {subjects}")

for subject in tqdm(sorted(subjects), desc='Subjects'):
    subject_unit_files = [f for f in unit_info_files if subject in f]
    subject_psth_files = [f for f in psth_files if subject in f]
    
    all_unit_dfs = []
    all_psth_data = []
    
    for unit_file in tqdm(sorted(subject_unit_files), desc=subject, leave=False):
        with open(os.path.join(unit_info_folder, unit_file), 'rb') as f:
            unit_df = pickle.load(f)
        all_unit_dfs.append(unit_df)
        
        psth_file = unit_file.replace('_unit_info.pkl', '_psth.npy')
        psth_path = os.path.join(psth_folder, psth_file)
        if os.path.exists(psth_path):
            psth_data = np.load(psth_path)
            all_psth_data.append(psth_data)
    
    combined_unit_df = pd.concat(all_unit_dfs, ignore_index=True)
    combined_psth = np.concatenate(all_psth_data, axis=0)
    
    print(f"  Combined: {len(combined_unit_df)} neurons")
    print(f"  Combined PSTH shape: {combined_psth.shape}")
    
    subject_filename = subject[0].upper() + subject[1:].lower()
    
    unit_output_file = os.path.join(output_folder, f"{subject_filename}_unit_info.pkl")
    with open(unit_output_file, 'wb') as f:
        pickle.dump(combined_unit_df, f)
    print(f"  Saved: {unit_output_file}")
    
    psth_output_file = os.path.join(output_folder, f"{subject_filename}_psth.npy")
    np.save(psth_output_file, combined_psth)
    print(f"  Saved: {psth_output_file}")

print("\nDone!")

Found 5 subjects: {'JianJian', 'MaoDan', 'TuTu', 'FaCai', 'ZhuangZhuang'}


Subjects:   0%|          | 0/5 [00:00<?, ?it/s]

  Combined: 2504 neurons
  Combined PSTH shape: (2504, 1072, 450)
  Saved: /media/ubuntu/sda/TrippleN/customize/Facai_unit_info.pkl


Subjects:  20%|██        | 1/5 [00:53<03:35, 53.80s/it]

  Saved: /media/ubuntu/sda/TrippleN/customize/Facai_psth.npy


...

Subjects:  80%|████████  | 4/5 [04:09<00:52, 52.98s/it]

  Saved: /media/ubuntu/sda/TrippleN/customize/Tutu_psth.npy


  Combined: 5025 neurons
  Combined PSTH shape: (5025, 1072, 450)
  Saved: /media/ubuntu/sda/TrippleN/customize/Zhuangzhuang_unit_info.pkl


Subjects: 100%|██████████| 5/5 [06:24<00:00, 76.86s/it]

  Saved: /media/ubuntu/sda/TrippleN/customize/Zhuangzhuang_psth.npy

Done!


In [10]:
import pickle
import warnings
import os
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

base_path = "/media/ubuntu/sda/TrippleN"
output_folder = os.path.join(base_path, "customize")

subjects = ['Jianjian', 'Facai', 'Tutu', 'Zhuangzhuang', 'Maodan']
subjects_lower = [s.lower() for s in subjects]

all_unit_dfs = []
all_psth_data = []

print("Combining all subjects...")
print("="*60)

for subject in tqdm(subjects, desc='Combining'):
    unit_file = os.path.join(output_folder, f"{subject}_unit_info.pkl")
    psth_file = os.path.join(output_folder, f"{subject}_psth.npy")
    
    if os.path.exists(unit_file):
        with open(unit_file, 'rb') as f:
            unit_df = pickle.load(f)
        
        # 筛选：删除best_r_time1等于best_r_time2的神经元
        valid_mask = unit_df['best_r_time1'] != unit_df['best_r_time2']
        filtered_unit_df = unit_df[valid_mask].reset_index(drop=True)
        
        print(f"  {subject}: {len(unit_df)} neurons → {len(filtered_unit_df)} neurons (removed {len(unit_df) - len(filtered_unit_df)})")
        
        all_unit_dfs.append(filtered_unit_df)
    
    if os.path.exists(psth_file):
        psth_data = np.load(psth_file)
        # 需要使用与unit_df相同的mask来筛选PSTH
        if len(all_unit_dfs) > 0:
            # 重新加载并筛选以确保对齐
            with open(unit_file, 'rb') as f:
                unit_df_orig = pickle.load(f)
            valid_mask = unit_df_orig['best_r_time1'] != unit_df_orig['best_r_time2']
            filtered_psth = psth_data[valid_mask]
            all_psth_data.append(filtered_psth)
        else:
            all_psth_data.append(psth_data)

combined_unit_df = pd.concat(all_unit_dfs, ignore_index=True)
combined_psth = np.concatenate(all_psth_data, axis=0)

print("="*60)
print(f"Combined all subjects:")
print(f"  Total neurons: {len(combined_unit_df)}")
print(f"  Combined PSTH shape: {combined_psth.shape}")

invalid_count = np.sum(combined_unit_df['best_r_time1'].values == combined_unit_df['best_r_time2'].values)
if invalid_count > 0:
    print(f"  WARNING: {invalid_count} neurons still have invalid time windows!")
else:
    print(f"  All neurons have valid time windows")

unit_output_file = os.path.join(output_folder, "all_subjects_unit_info.pkl")
with open(unit_output_file, 'wb') as f:
    pickle.dump(combined_unit_df, f)
print(f"  Saved: {unit_output_file}")

psth_output_file = os.path.join(output_folder, "all_subjects_psth.npy")
np.save(psth_output_file, combined_psth)
print(f"  Saved: {psth_output_file}")


print("\nDone!")

Combining all subjects...


Combining:   0%|          | 0/5 [00:00<?, ?it/s]

  Jianjian: 6180 neurons → 6167 neurons (removed 13)


Combining:  20%|██        | 1/5 [00:15<01:03, 15.92s/it]

  Facai: 2504 neurons → 2499 neurons (removed 5)


...

  Zhuangzhuang: 5025 neurons → 5018 neurons (removed 7)


Combining:  80%|████████  | 4/5 [01:32<00:17, 17.99s/it]

  Maodan: 1122 neurons → 1112 neurons (removed 10)


Combining: 100%|██████████| 5/5 [01:44<00:00, 20.82s/it]


Combined all subjects:
  Total neurons: 15652
  Combined PSTH shape: (15652, 1072, 450)
  All neurons have valid time windows
  Saved: /media/ubuntu/sda/TrippleN/customize/all_subjects_unit_info.pkl
  Saved: /media/ubuntu/sda/TrippleN/customize/all_subjects_psth.npy

Done!


In [11]:
import pickle
with open('/media/ubuntu/sda/TrippleN/customize/aggregate_response/all_subjects_unit_info.pkl', 'rb') as f:
    unit_df = pickle.load(f)

In [16]:
import accelerate

In [17]:
import transformers

In [18]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

In [25]:
tokenizer = T5Tokenizer.from_pretrained("/media/ubuntu/sda/TrippleN/model/flantt5")
model = T5ForConditionalGeneration.from_pretrained("/media/ubuntu/sda/TrippleN/model/flantt5", ignore_mismatched_sizes = True)

Loading weights: 100%|██████████| 560/560 [00:00<00:00, 2833.29it/s, Materializing param=shared.weight]                                                       
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [26]:
input_text = 'Translate English to German: How old are you?'
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

output = model.generate(input_ids)

In [28]:
print(tokenizer.decode(output[0]))

<pad> Wie alt sind Sie?</s>
